In [18]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
import mediapipe.python.solutions.hands as mp_hands
import mediapipe.python.solutions.drawing_utils as drawing
import mediapipe.python.solutions.drawing_styles as drawing_styles
import PIL.Image as pilimg
from PIL import ImageTk

In [19]:
model = load_model("rps_cnn.keras")
classes = ["Paper","Rock","Scissors"]
hands = mp_hands.Hands(
    static_image_mode=False,  # Set to False for processing video frames
    max_num_hands=2,           # Maximum number of hands to detect
    min_detection_confidence=0.5  # Minimum confidence threshold for hand detection
)

In [20]:
def video():
    global pred
    success,frame = camera.read()
    if success:
        h,w,c = frame.shape
        pFrame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        hands_detected = hands.process(pFrame)
        if hands_detected.multi_hand_landmarks:
                hand = hands_detected.multi_hand_landmarks[0]
                drawing.draw_landmarks(frame,hand,mp_hands.HAND_CONNECTIONS)
                x = [landmark.x for landmark in hand.landmark]
                y = [landmark.y for landmark in hand.landmark]
                center = np.array([np.mean(x)*w, np.mean(y)*h]).astype('int32')
                xTL = center[0]-100
                yTL = center[1]-100
                xBR = center[0]+100
                yBR = center[1]+100
                xTL = max(0, xTL)
                yTL = max(0, yTL)
                xBR = min(w, xBR)
                yBR = min(h, yBR)
        
                cropFrame = frame[yTL:yBR, xTL:xBR]
                cv2.rectangle(frame, (xTL,yTL), (xBR,yBR), (0,0,255), 1)
                if cropFrame.size != 0:
                    newFrame = cv2.cvtColor(cropFrame, cv2.COLOR_BGR2RGB)
                    newFrame = cv2.resize(newFrame,(128,128))
                    newFrame = newFrame/255
                    newFrame = np.expand_dims(newFrame,axis=0)
                    prediction = model.predict(newFrame)
                    predict = classes[np.argmax(prediction[0])]
                    cv2.putText(frame,f"{predict}",(center[0]-80,center[1]-120),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)
                if predict:
                    lblMove.config(text=f"Detected {predict}")
                    pred = predict
        guiFrame = cv2.resize(pFrame,(70,30))
        guiFrame = pilimg.fromarray(guiFrame)
        guiFrame = ImageTk.PhotoImage(image=guiFrame)
        webcam.config(image=guiFrame)
        webcam.image = guiFrame
    main.after(10,video)

def end():
    camera.release()
    main.destroy()

In [21]:
from tkinter import *
from tkinter import messagebox
import random
import tkinter.font as f

playerScore = 0
compScore = 0
draws = 0
pred = None

def compChoose():
    choice = random.randint(1,3)
    if choice == 1:
        choice = "Rock"
    elif choice == 2:
        choice = "Paper"
    else:
        choice = "Scissors"
    return choice

def start(pred):
    if pred:
        answer = messagebox.askyesno("Confirm", f"The model detected {pred}.\n\nIs this correct?")
        if answer == 1:
            play(pred)
        else:
            lblRes.config(text="Adjust your hand. \nTry again.")
            lblRes.config(foreground="dark blue")
    else:
        messagebox.showwarning("ERROR","No hand detected. Please try again.")

def play(playerChoice):
    global playerScore, compScore, draws
    btnStart.config(text="Play Again")
    compChoice = compChoose()
    if playerChoice == compChoice:
        result = "DRAW!"
        draws += 1

    elif playerChoice == "Rock" and compChoice == "Scissors":
        result = "YOU WIN!"
        playerScore += 1
    elif playerChoice == "Paper" and compChoice == "Rock":
        result = "YOU WIN!"
        playerScore += 1
    elif playerChoice == "Scissors" and compChoice == "Paper":
        result = "YOU WIN!"
        playerScore += 1

    else:
        result = "YOU LOSE!"
        compScore += 1

    update(playerChoice, compChoice,result)

def update(pc,cc,result):
    lblRes.config(
        text=("{}\n\nYou chose: \n{}\nComputer chose: \n{}".format(result,pc,cc))
    )
    if result == "YOU WIN!":
        lblRes.config(foreground="green")
    elif result == "YOU LOSE!":
        lblRes.config(foreground="red")
    else:
        lblRes.config(foreground="dark blue")
    lblPS.config(text=f"Player score: {playerScore}")
    lblCS.config(text=f"Computer score: {compScore}")
    lblD.config(text=f"Draws: {draws}")
    lblMove.config(text=f"Detected {pred}")

def reset():
    global playerScore, compScore, draws

    playerScore = 0
    compScore = 0
    draws = 0

    lblPS.config(text="Player score: 0")
    lblCS.config(text="Computer score: 0")
    lblD.config(text="Draws: 0")
    lblRes.config(text="Start a new round!")
    lblRes.config(foreground="dark blue")
    btnStart.config(text="Start Round")


main = Tk()
main.title("Rock Paper Scissors")
main.geometry("800x600")

fontR = f.Font(family="Calibri",size=14,weight="bold")
fontT = f.Font(family="Calibri",size=22,weight="bold")

lblT = Label(main,text="Rock Paper Scissors",font=fontT,foreground="dark blue")
lblT.grid(row=0,column=0,columnspan=2,pady=3)

webcam = Label(main,width=70,height=30,background="black")
webcam.grid(row=1,column=0,rowspan=6,padx=20,pady=3)

lblMove = Label(main,text=f"Detected {pred}",font=fontR,foreground="dark blue")
lblMove.grid(row=1,column=1,padx=10,pady=3)

btnStart = Button(main,text="Start Round",command=lambda:start(pred),font=fontR,width=18)
btnStart.grid(row=2,column=1,padx=10,pady=3)

lblRes = Label(main,text="Ready to play!",font=fontR,foreground="dark blue")
lblRes.grid(row=3,column=1,padx=10,pady=3)

lblPS = Label(main,text="Player score: 0",font=fontR)
lblPS.grid(row=4,column=1,padx=10,pady=3)

lblCS = Label(main,text="Computer score: 0",font=fontR)
lblCS.grid(row=5,column=1,padx=10,pady=3)

lblD = Label(main,text="Draws: 0",font=fontR)
lblD.grid(row=6,column=1,padx=10,pady=3)

btnReset = Button(main,text="Reset",command=reset,font=fontR)
btnReset.grid(row=7,column=0,pady=15)

btnEnd = Button(main,text="End Game",command=end,font=fontR)
btnEnd.grid(row=7,column=1,pady=15)

camera = cv2.VideoCapture(0)
video()
main.mainloop()

TclError: image "pyimage1" doesn't exist

I have created a rock paper scissors game utilising realtime object detection and deep learning models. The project takes feed from a camera and uses transfer learning of a CNN model to classify the gesture into rock, paper or scissors. The GUI is built using the tkinter library and allows the user to play against the computer, with results and scores available. 

In [ ]:
print(Image)
print(hasattr(Image, "fromarray"))
print(Image.__file__)
print(type(Image))

<class 'tkinter.Image'>
False


AttributeError: type object 'Image' has no attribute '__file__'